# Socio-Economic, Institutional, and Experiential Determinants of Knowledge Management Practices Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the [FAIR^2](https://sen.science/doi/10.71728/senscience.xhws-jsxj) dataset using the `mlcroissant` library. You will:
* Load data and metadata via the Croissant schema
* Explore record sets and fields (referenced by their `@id`)
* Extract and analyze tabular data
* Perform simple exploratory data analysis (EDA) and visualize key variables

### Dataset Source
*Croissant schema URL*: https://sen.science/doi/10.71728/senscience.xhws-jsxj/fair2.json

In [ ]:
# Ensure the latest mlcroissant library is installed
!pip install -U mlcroissant

## 1. Data Loading
We start by loading the dataset via the Croissant schema. The `mlcroissant.Dataset` class provides methods to access metadata, record sets, and records.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.xhws-jsxj/fair2.json"

# Load dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
# Print summary information
print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"Date Published: {metadata.datePublished}")
print(f"Version: {metadata.version}")
print(f"Keywords: {', '.join(metadata.keywords)}")
print(f"License: {metadata.license}")

## 2. Data Overview
Here we list all available record sets (`cr:RecordSet`) including their `@id` and a short overview of their fields, referencing all entities by their `@id` as required.

> **Tip:** Use record set and field `@id` for all programmatic extraction and analysis.

In [ ]:
# Retrieve record sets
from pprint import pprint

record_sets = list(dataset.record_sets)  # Each is a mlc.RecordSet object
print(f"Number of record sets: {len(record_sets)}\n")
all_recordset_ids = [rs.id for rs in record_sets]
for rs in record_sets:
    print(f"Record set @id: {rs.id}")
    print(f"  Name: {rs.name}")
    # List fields in this record set
    print(f"  Fields:")
    for field in rs.fields:
        print(f"    - @id: {field.id}, Name: {field.name}, DataType: {field.data_type}")
    print()
# If the dataset has no record sets, mlcroissant provides a default one (often from tabular files)

if len(record_sets) == 0:
    print("No specific record sets defined; dataset likely contains a single table. Reading default records...")
    # Print the columns available
    sample_records = list(dataset.records(record_set=None, limit=1))
    if sample_records:
        print(f"Fields (columns): {list(sample_records[0].keys())}")

## 3. Data Extraction
Here we extract tabular data from the main record set. We'll use the discovered `@id` value of the primary record set (or `None` if a default is provided). All further references to fields are made by their `@id`.

In [ ]:
# Choose a record set to extract (use the @id from the previous cell output; if none, use None)
if len(record_sets) > 0:
    record_set_id = record_sets[0].id  # Use first record set as example
else:
    record_set_id = None

# Load all records for the selected record set
records = list(dataset.records(record_set=record_set_id))
df = pd.DataFrame(records)
print(f"Loaded {len(df)} records with columns: {df.columns.tolist()}")
df.head()

## 4. Exploratory Data Analysis (EDA)
Let's try some filtering, normalization, and simple grouping using only column (field) `@id`s as discovered above. We'll choose one numeric field and one group field for demonstration.

In [ ]:
# For demonstration, pick a numeric field and a grouping field by @id (adjust as needed)
# List numeric candidates
numeric_candidates = []
group_candidates = []
if len(record_sets) > 0:
    # Look at all fields' @id and data_type
    for field in record_sets[0].fields:
        if field.data_type in ['Float', 'Number', 'Integer'] and field.id in df.columns:
            numeric_candidates.append(field.id)
        if field.data_type == 'Text' and field.id in df.columns:
            group_candidates.append(field.id)

print("Available numeric field ids:", numeric_candidates)
print("Available group field ids:", group_candidates)

# Pick the first numeric and group field found
if numeric_candidates:
    numeric_field = numeric_candidates[0]
else:
    numeric_field = df.select_dtypes(include=['number']).columns[0] if len(df.select_dtypes(include=['number']).columns) > 0 else None
if group_candidates:
    group_field = group_candidates[0]
else:
    group_field = df.select_dtypes(include=['object']).columns[0] if len(df.select_dtypes(include=['object']).columns) > 0 else None

if numeric_field:
    print(f'Using numeric field @id for filtering and normalization: {numeric_field}')
    threshold = df[numeric_field].mean()  # Use mean as demonstration threshold
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered {len(filtered_df)} records where {numeric_field} > {threshold:.2f} (mean value)")

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (
        (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    )
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by group_field if available
    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped mean of {numeric_field} by {group_field}:")
        print(grouped_df.head())
else:
    print("No numeric field found for demo. Please examine df.columns and select manually.")

## 5. Visualization
We now visualize the distribution of the selected numeric field, and optionally plot means by the group field (if found).

> This step requires `matplotlib` and/or `seaborn`. We install them if necessary.

In [ ]:
# Install plotting libraries if needed
!pip install -q matplotlib seaborn

import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field], kde=True)
    plt.title(f"Distribution of field: {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

if numeric_field and group_field and group_field in df.columns:
    # Show group means (if a manageable number of groups)
    gp = df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False)
    if len(gp) < 25:
        plt.figure(figsize=(10,5))
        sns.barplot(x=gp.index, y=gp.values)
        plt.title(f"Mean of {numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we've demonstrated how to:
- Programmatically explore a Croissant dataset using `mlcroissant`
- Work with metadata, record sets, and fields by their `@id`
- Extract records into a DataFrame, filter, normalize, and group data via field `@id`
- Visualize value distributions and grouped means

For more advanced analysis, continue using field and record set `@id` for all operations to ensure reproducibility and correctness. Refer to the [mlcroissant documentation](https://mlcommons.org/croissant/) for further utilities and schema details.